# Unidade III — Pré-processamento de Dados

## Qualidade, limpeza e integração

**Carga estimada:** 3 horas  
**Pré-requisitos:** pandas, estatística descritiva e tipos de atributos.

> **Pergunta norteadora:** como transformar registros problemáticos em uma base analisável sem apagar evidências nem inventar certezas?


## Objetivos de aprendizagem

Ao concluir este notebook, você será capaz de:

- diagnosticar completude, validade, consistência, unicidade e atualidade;
- tratar ausências, duplicatas e valores inválidos com regras justificadas;
- integrar tabelas controlando cardinalidade e correspondência de entidades;
- comparar indicadores antes e depois sem modificar os dados brutos.


In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## Qualidade depende do uso

Qualidade não significa perfeição abstrata. Um dado é adequado quando sustenta a finalidade declarada. **Completude** examina ausências; **validade**, conformidade com domínios e formatos; **consistência**, compatibilidade entre campos e fontes; **unicidade**, registros repetidos; e **atualidade**, adequação temporal. Acurácia, isto é, proximidade do valor real, frequentemente exige uma fonte externa confiável.

O exemplo abaixo é sintético e introduz problemas conhecidos para que possamos verificar o efeito de cada decisão. A cópia `clientes_brutos` será preservada; toda alteração ocorrerá em outra tabela.


In [2]:
clientes_brutos = pd.DataFrame({
    "cliente_id": [101, 102, 103, 104, 104, 105, 106, 107],
    "idade": [34, np.nan, 29, 150, 150, 45, 38, 52],
    "cidade": ["Recife", " recife " , "Olinda", "Recife", "Recife", "Paulista", "OLINDA", "Recife"],
    "email": ["ana@exemplo.br", "BRUNO@EXEMPLO.BR", None, "dora@exemplo.br", "dora@exemplo.br", "eva@exemplo.br", "f@exemplo.br", "g@exemplo.br"],
    "mensalidade": [89.9, 110.0, np.nan, 95.0, 95.0, -20.0, 130.0, 120.0],
})
clientes_brutos


,cliente_id,idade,cidade,email,mensalidade
0,101,34.0,Recife,ana@exemplo.br,89.9
1,102,NaN,recife,BRUNO@EXEMPLO.BR,110.0
2,103,29.0,Olinda,None,NaN
3,104,150.0,Recife,dora@exemplo.br,95.0
4,104,150.0,Recife,dora@exemplo.br,95.0
5,105,45.0,Paulista,eva@exemplo.br,-20.0
6,106,38.0,OLINDA,f@exemplo.br,130.0
7,107,52.0,Recife,g@exemplo.br,120.0


## Diagnóstico antes da correção

O relatório torna os critérios auditáveis. Ausência é medida por coluna; duplicidade usa a chave de entidade; validade requer regras do domínio. Aqui, idades devem estar entre 18 e 100 anos e mensalidades devem ser positivas. Esses limites são hipóteses do estudo, não regras universais.


In [3]:
def relatorio_qualidade(df: pd.DataFrame) -> pd.Series:
    """Resume problemas definidos para este estudo de caso."""
    return pd.Series({
        "linhas": len(df),
        "celulas_ausentes": int(df.isna().sum().sum()),
        "ids_duplicados": int(df.duplicated("cliente_id", keep=False).sum()),
        "idades_invalidas": int((~df["idade"].between(18, 100) & df["idade"].notna()).sum()),
        "mensalidades_invalidas": int((df["mensalidade"] <= 0).sum()),
    })

antes = relatorio_qualidade(clientes_brutos)
antes.to_frame("antes")


,antes
linhas,8
celulas_ausentes,3
ids_duplicados,2
idades_invalidas,2
mensalidades_invalidas,1


## Limpeza rastreável

A ordem importa: primeiro padronizamos texto; depois marcamos valores impossíveis como ausentes; em seguida consolidamos duplicatas exatas de entidade; por fim imputamos medianas. Imputação reduz ausências, mas não recupera o valor verdadeiro e pode reduzir artificialmente a variabilidade. Em modelagem supervisionada, seus parâmetros devem ser aprendidos apenas no treino, como veremos no próximo notebook.


In [4]:
clientes_limpos = clientes_brutos.copy(deep=True)
clientes_limpos["cidade"] = clientes_limpos["cidade"].str.strip().str.title()
clientes_limpos["email"] = clientes_limpos["email"].str.strip().str.lower()
clientes_limpos.loc[~clientes_limpos["idade"].between(18, 100), "idade"] = np.nan
clientes_limpos.loc[clientes_limpos["mensalidade"] <= 0, "mensalidade"] = np.nan
clientes_limpos = clientes_limpos.drop_duplicates("cliente_id", keep="first")
for coluna in ["idade", "mensalidade"]:
    clientes_limpos[coluna] = clientes_limpos[coluna].fillna(clientes_limpos[coluna].median())

pd.testing.assert_frame_equal(clientes_brutos, clientes_brutos.copy())
clientes_limpos


,cliente_id,idade,cidade,email,mensalidade
0,101,34.0,Recife,ana@exemplo.br,89.9
1,102,38.0,Recife,bruno@exemplo.br,110.0
2,103,29.0,Olinda,None,110.0
3,104,38.0,Recife,dora@exemplo.br,95.0
5,105,45.0,Paulista,eva@exemplo.br,110.0
6,106,38.0,Olinda,f@exemplo.br,130.0
7,107,52.0,Recife,g@exemplo.br,120.0


## Integração e cardinalidade

Integração não é apenas chamar `merge`. Precisamos definir a entidade, harmonizar chaves e declarar a cardinalidade esperada. A opção `validate="one_to_one"` faz a operação falhar se qualquer tabela tiver mais de uma linha por cliente. Em dados reais, nomes ou e-mails aproximados exigem resolução de entidades, revisão de falsos pares e proteção de dados pessoais.


In [5]:
contratos = pd.DataFrame({
    "cliente_id": [101, 102, 103, 104, 105, 106, 108],
    "plano": ["Básico", "Pro", "Básico", "Pro", "Básico", "Pro", "Básico"],
})

integrados = clientes_limpos.merge(
    contratos, on="cliente_id", how="left", validate="one_to_one", indicator=True
)
integrados[["cliente_id", "plano", "_merge"]]


,cliente_id,plano,_merge
0,101,Básico,both
1,102,Pro,both
2,103,Básico,both
3,104,Pro,both
4,105,Básico,both
5,106,Pro,both
6,107,NaN,left_only


In [6]:
depois = relatorio_qualidade(clientes_limpos)
comparacao = pd.concat([antes.rename("antes"), depois.rename("depois")], axis=1)
comparacao["variacao"] = comparacao["depois"] - comparacao["antes"]
comparacao


,antes,depois,variacao
linhas,8,7,-1
celulas_ausentes,3,1,-2
ids_duplicados,2,0,-2
idades_invalidas,2,0,-2
mensalidades_invalidas,1,0,-1


O relatório mostra que duplicidades e violações de domínio foram eliminadas segundo as regras declaradas. Uma ausência permanece no e-mail porque não existe base defensável para inventar esse identificador. A redução dos indicadores não prova acurácia: valores plausíveis ainda podem estar errados. A junção também revela clientes sem contrato correspondente; eles devem ser investigados, e não removidos silenciosamente.

> **U03-NB01-V01 — Verifique seu entendimento:** por que substituir uma idade impossível pela mediana não torna esse valor conhecido nem garante acurácia?

> **U03-NB01-E01 — Exercício:** adapte `relatorio_qualidade` para incluir a proporção de ausências por coluna e o número de clientes sem contrato. Entregue a função, a tabela resultante e duas interpretações.


## Síntese

- Qualidade é definida em relação ao uso e ao domínio.
- Dados brutos devem permanecer imutáveis; transformações precisam ser rastreáveis.
- Imputação expressa uma decisão sob incerteza.
- Junções exigem chave, cardinalidade e auditoria de correspondências.

## Referências

- HAN, Jiawei; PEI, Jian; TONG, Hanghang. *Data Mining: Concepts and Techniques*. 4. ed. Cambridge: Morgan Kaufmann/Elsevier, 2023. Cap. 2, seção 2.4.
- PANDAS DEVELOPMENT TEAM. *pandas documentation*: missing data e merge. Versão 2.x.
